In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from datetime import datetime, timedelta

from core.portfolio.backtest import BacktestEngine
from core.portfolio.asset import Portfolio
from core.data.instrument.provider import InstrumentProvider
from core.data.instrument.instrument import Instrument
from core.ml.pricedir import PriceDirPredictorRF, PriceDirPredictorLGBM, PriceDirPredictorXGB, PriceDirPredictorEnsemble

ip = InstrumentProvider()
instrument: Instrument = ip.get_instrument('NVDA')

HORIZON = 5                     # Prediction horizon (days)
RISK_PER_TRADE_PCT = 0.02       # Max portfolio risk per trade (2%)
MAX_PORTFOLIO_CAP_PCT = 0.40    # Hard cap on single position allocation (40% max cash)
STOP_LOSS_ATR_MULT = 2.0

predictor = PriceDirPredictorLGBM(instrument=instrument, horizon_days=HORIZON)

portfolio = Portfolio('BT1', 'USD')
start_date = datetime(2025, 1, 1)
engine = BacktestEngine(portfolio, start_date=start_date, initial_cash=1000.0)

days_since_last_train = HORIZON  

position_tracker = {
    'entry_price': None,
    'stop_loss_price': None
}

def ml_strategy_with_stop_loss():
    global days_since_last_train, position_tracker

    current_date = engine.date

    if days_since_last_train >= HORIZON:
        training_cutoff = current_date - timedelta(days=HORIZON)
        train_stats = predictor.train(end_date=training_cutoff)
        print(f"\n[RETRAINED {current_date.strftime('%Y-%m-%d')}] "
              f"Cutoff: {training_cutoff.strftime('%Y-%m-%d')} | "
              f"Acc: {train_stats['train_accuracy']:.2%}")
        days_since_last_train = 0

    forecast = predictor.predict_at_date(as_of_date=current_date)
    signal = forecast['signal']
    confidence = forecast['confidence_up']
    atr = forecast['atr']
    close_price = forecast['close']

    has_position = any(a.symbol == instrument.symbol and a.volume > 0 for a in portfolio.assets)

    if has_position and position_tracker['stop_loss_price'] is not None:
        stop_price = position_tracker['stop_loss_price']
        
        if close_price <= stop_price:
            engine.sell(symbol=instrument.symbol, amount=engine.cash)
            print(
                f"  [{current_date.strftime('%Y-%m-%d')}] 🛑 STOP-LOSS TRIGGERED | "
                f"Close: ${close_price:.2f} <= Stop: ${stop_price:.2f} | "
                f"Entry: ${position_tracker['entry_price']:.2f}"
            )
            position_tracker['entry_price'] = None
            position_tracker['stop_loss_price'] = None
            days_since_last_train += 1
            return

    if signal == 1 and confidence > 0.55 and not has_position:
        if atr > 0 and close_price > 0:
            total_portfolio_value = engine.total_value
            risk_budget = total_portfolio_value * RISK_PER_TRADE_PCT
            atr_stop_distance = STOP_LOSS_ATR_MULT * atr
            
            target_shares = risk_budget / atr_stop_distance
            calculated_amount = target_shares * close_price

            max_cash_allowed = engine.cash * MAX_PORTFOLIO_CAP_PCT
            final_allocation = min(calculated_amount, max_cash_allowed)

            if final_allocation > 50:
                engine.buy(instrument, amount=final_allocation)
                
                position_tracker['entry_price'] = close_price
                position_tracker['stop_loss_price'] = close_price - atr_stop_distance
                
                print(
                    f"  [{current_date.strftime('%Y-%m-%d')}] BUY {instrument.symbol} | "
                    f"Entry: ${close_price:.2f} | Stop: ${position_tracker['stop_loss_price']:.2f} | "
                    f"Conf: {confidence:.2%}"
                )

    elif signal == 0 and has_position:
        engine.sell(symbol=instrument.symbol, amount=engine.cash)
        print(f"  [{current_date.strftime('%Y-%m-%d')}] SELL Executed (ML Signal) | Conf DOWN: {forecast['confidence_down']:.2%}")
        
        position_tracker['entry_price'] = None
        position_tracker['stop_loss_price'] = None

    days_since_last_train += 1

print("--- Backtest ---")
reached_present = False
i = 0
while not reached_present:
    reached_present = engine.next_day(strategy=ml_strategy_with_stop_loss)
    print(f"Day {i+1} ({engine.date.strftime('%Y-%m-%d')}): Assets Value: ${engine.assets_values:,.2f}, Cash: ${engine.cash}, Total Value: ${engine.total_value}")
    i += 1

engine.show_transaction_log_df()

--- Backtest ---

[RETRAINED 2025-01-01] Cutoff: 2024-12-27 | Acc: 65.36%
Day 1 (2025-01-02): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 2 (2025-01-03): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 3 (2025-01-04): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 4 (2025-01-05): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 5 (2025-01-06): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0

[RETRAINED 2025-01-06] Cutoff: 2025-01-01 | Acc: 64.84%
Day 6 (2025-01-07): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
  [2025-01-07] BUY NVDA | Entry: $139.93 | Stop: $128.18 | Conf: 55.98%
Day 7 (2025-01-08): Assets Value: $238.07, Cash: $761.8811370882529, Total Value: $999.948949003878
Day 8 (2025-01-09): Assets Value: $230.94, Cash: $761.8811370882529, Total Value: $992.8188265455044
Day 9 (2025-01-10): Assets Value: $230.94, Cash: $761.8811370882529, Total Value: $992.8188265455044
Day 10 (2025-01-11): Assets Value

Type,Symbol,Volume,Price ($),Total Amount ($),Fee ($)
BUY,NVDA,1.701700,139.93,238.12,0.00
SELL,NVDA,1.701700,133.03,226.38,0.00
BUY,NVDA,1.287089,123.52,158.98,0.00
SELL,NVDA,1.287089,133.37,171.66,0.00
BUY,NVDA,1.472895,119.97,176.70,0.00
SELL,NVDA,1.472895,115.27,169.78,0.00
BUY,NVDA,1.571213,118.36,185.97,0.00
SELL,NVDA,1.571213,117.54,184.68,0.00
BUY,NVDA,1.731398,120.52,208.67,0.00
SELL,NVDA,1.731398,113.60,196.69,0.00
